In [2]:
pip install xgboost prettytable gpxpy pandarallel numpy pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans, KMeans
import gpxpy.geo # Get the haversine distance
from sklearn.linear_model import LinearRegression
from sklearn import tree
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import math
from prettytable import PrettyTable

In [ ]:

base_path = "./"
# we speed the process by decreasing the dimensionality
columns=['tpep_pickup_datetime',
           'tpep_dropoff_datetime',
           'trip_distance',
           'pickup_longitude',
           'pickup_latitude',
           'dropoff_longitude',
           'dropoff_latitude',
           'total_amount']

df_2016_1 = pd.read_csv(f'{base_path}yellow_tripdata_2016-01.csv', usecols=columns, nrows=1000000)
df_2016_2 = pd.read_csv(f'{base_path}yellow_tripdata_2016-02.csv', usecols=columns, nrows=1000000)
df_2016_3 = pd.read_csv(f'{base_path}yellow_tripdata_2016-03.csv', usecols=columns, nrows=1000000)

df = df_2016_1.append(df_2016_2).append(df_2016_3)

original_len = df.shape[0]
print(original_len)

1000000


Подготовка данных, создадим новые фичи для модели.

In [4]:

def clean_data(df, test=False, predict=False):
    df = df.dropna(how='any', axis='rows')
    df = df[(df.dropoff_latitude != 0) | (df.dropoff_longitude != 0)]
    df = df[(df.pickup_latitude != 0) | (df.pickup_longitude != 0)]
    
    if "total_amount" in list(df):
        df = df[df.total_amount.between(5, 45)]
    
    return df

df = clean_data(df)

### 2. Уберем поздки более 3 часов и удалим ошибочные данные (где время меньше 0).

In [5]:
def clean_trip_duration(df):

    df['tpep_pickup_datetime']  = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime']  = pd.to_datetime(df['tpep_dropoff_datetime'])

    trip_duration = np.array(df['tpep_dropoff_datetime']-df['tpep_pickup_datetime'])
    trip_duration = trip_duration/1000000000/60
    df['trip_duration'] = trip_duration.astype(float)
    
    df.drop(df[(df['trip_duration'] > 180) | 
               (df['trip_duration'] <= 0)].index, inplace = True)


clean_trip_duration(df)

### 3. Переименуем переменную

In [6]:
def clean_pickuptime(df):
    return df.rename(columns={'tpep_pickup_datetime': 'pickup_time'})

df = clean_pickuptime(df)

### 4. Переведем мили в километры и уберем значения где у поездок расстояние менее 0 и более 40 км.

In [7]:
def clean_trip_distance(df):
    df['trip_distance']=df['trip_distance']*1.61
    df.drop(df[(df['trip_distance'] <= 0) | (df['trip_distance'] > 40)].index, inplace = True)
    
# clean_trip_distance(df_2015)
clean_trip_distance(df)

### 5. Посчитаем скорость и уберем аномальные данные

In [8]:
def compute_speed(df):
    df['speed'] = df['trip_distance']/df['trip_duration']*60
    
def clean_speed(df):

    df.drop(df[((df['speed'] <= 0) | (df['speed'] > 120))].index, inplace = True)


compute_speed(df)    
clean_speed(df)



### 6. Определим кластеры по региону поездок


In [9]:
from datetime import datetime, timedelta
from sklearn.cluster import MiniBatchKMeans, KMeans
from pandarallel import pandarallel


coord = df[["pickup_latitude", "pickup_longitude"]].values
regions = MiniBatchKMeans(n_clusters = 30, batch_size = 10000).fit(coord)

cluster_column = regions.predict(df[["pickup_latitude", "pickup_longitude"]])
df["pickup_cluster"] = cluster_column

c:\Users\user\MyProjects\MFDP\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MiniBatchKMeans was fitted without feature names
  warnings.warn(


In [10]:
from pandarallel import pandarallel
import pandas as pd

pandarallel.initialize()  # You can also set nb_workers if needed

def process_time(x):
    import pandas as pd
    from datetime import timedelta
    return pd.to_datetime(x).replace(minute=0, second=0) + timedelta(hours=1)

df['pickup_time'] = df['pickup_time'].parallel_apply(process_time)

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


Соберем обучающий датасет: 

Зависимая переменная(спрос) - количество поездок (cout)
Объясняющие переменные - день, месяц, день недели, час

In [11]:
from pandarallel import pandarallel
from functools import partial
import pandas as pd

# Initialize pandarallel
pandarallel.initialize()

# Groupby and reset index
df = df.groupby(['pickup_time', 'pickup_cluster']).size().reset_index(name='count')

# Compute max count
max_count = df['count'].max()

# Define the function explicitly with all needed variables
def normalize_count(x, max_count):
    return x / max_count

# Create a partially applied function with max_count baked in
func = partial(normalize_count, max_count=max_count)

# Apply in parallel
df['count'] = df['count'].parallel_apply(func)

# Extract datetime features
df['month'] = pd.DatetimeIndex(df['pickup_time']).month
df['day'] = pd.DatetimeIndex(df['pickup_time']).day
df['dayofweek'] = pd.DatetimeIndex(df['pickup_time']).dayofweek
df['hour'] = pd.DatetimeIndex(df['pickup_time']).hour

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


### 7. разделение на тестовую и обучающую выборки

In [12]:
from sklearn.model_selection import train_test_split
# training X and y
X = df[['pickup_cluster', 'month', 'day', 'hour', 'dayofweek']]
y = df['count']


X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.33, random_state=42
)

Линейная регрессия

In [13]:
LReg = LinearRegression()
LReg.fit(X_train, y_train)
LReg_y_pred = LReg.predict(X_test)

Случайный лес

In [14]:
RFRegr = RandomForestRegressor()
RFRegr.fit(X_train, y_train)
RFRegr_y_pred = RFRegr.predict(X_test)

XGBoost

In [15]:
GBRegr = XGBRegressor(n_estimators=1000, max_depth=7, eta=0.1, subsample=0.7, colsample_bytree=0.8)
GBRegr.fit(X_train, y_train)
GBRegr_y_pred = GBRegr.predict(X_test)

Оценка моделей

In [16]:
def model_evaluation(algorithem_name, X_Test, y_pred, y_true):
    
    # R2 and Adjasted R2
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1-(1-r2)*((len(X_Test)-1)/(len(X_Test)-X_Test.shape[1]-1))
    # MSE and RMSE
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    
    # print in table
    x = PrettyTable()
    x.add_row(['R2', r2])
    x.add_row(['Adjusted R2', adj_r2])
    x.add_row(['MSE',mse])
    x.add_row(['RMSE', rmse])
    x.title = algorithem_name
    print(x)
    


In [17]:
model_evaluation('y True',X_Test=X_test, y_pred=y_test, y_true=y_test)

+-----------------------+
|         y True        |
+-------------+---------+
|   Field 1   | Field 2 |
+-------------+---------+
|      R2     |   1.0   |
| Adjusted R2 |   1.0   |
|     MSE     |   0.0   |
|     RMSE    |   0.0   |
+-------------+---------+


### 2. Линейная регрессия

In [18]:
model_evaluation('Linear Regression',X_Test=X_test, y_pred=LReg_y_pred, y_true=y_test)

+-----------------------------------+
|         Linear Regression         |
+-------------+---------------------+
|   Field 1   |       Field 2       |
+-------------+---------------------+
|      R2     | 0.16772667518271622 |
| Adjusted R2 | 0.16166937296424988 |
|     MSE     |  0.024927445640402  |
|     RMSE    | 0.15788427926934967 |
+-------------+---------------------+


### 3. Случайный лес

In [19]:
model_evaluation('Random Forest',X_Test=X_test, y_pred=RFRegr_y_pred, y_true=y_test)

+------------------------------------+
|           Random Forest            |
+-------------+----------------------+
|   Field 1   |       Field 2        |
+-------------+----------------------+
|      R2     |  0.9207209709856217  |
| Adjusted R2 |  0.9201439765968707  |
|     MSE     | 0.002374488797431575 |
|     RMSE    | 0.04872872661409874  |
+-------------+----------------------+


### 4. XGBoost

In [20]:
model_evaluation('Gradient Boosting',X_Test=X_test, y_pred=GBRegr_y_pred, y_true=y_test)

+-------------------------------------+
|          Gradient Boosting          |
+-------------+-----------------------+
|   Field 1   |        Field 2        |
+-------------+-----------------------+
|      R2     |   0.9481223424201982  |
| Adjusted R2 |   0.9477447757711458  |
|     MSE     | 0.0015537894231511013 |
|     RMSE    |  0.039418135713794246 |
+-------------+-----------------------+


## Тест

In [21]:
LReg_y_pred = LReg.predict(X)
RFRegr_y_pred = RFRegr.predict(X)
GBRegr_y_pred = GBRegr.predict(X)

c:\Users\user\MyProjects\MFDP\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
c:\Users\user\MyProjects\MFDP\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


### Фактические

In [22]:
model_evaluation('y True',X_Test=X, y_pred=y, y_true=y)

+-----------------------+
|         y True        |
+-------------+---------+
|   Field 1   | Field 2 |
+-------------+---------+
|      R2     |   1.0   |
| Adjusted R2 |   1.0   |
|     MSE     |   0.0   |
|     RMSE    |   0.0   |
+-------------+---------+


### Линейная регрессия

In [23]:
model_evaluation('Linear Regression',X_Test=X, y_pred=LReg_y_pred, y_true=y)

+------------------------------------+
|         Linear Regression          |
+-------------+----------------------+
|   Field 1   |       Field 2        |
+-------------+----------------------+
|      R2     | 0.18519380793265672  |
| Adjusted R2 | 0.18324637439521085  |
|     MSE     | 0.024857460096273893 |
|     RMSE    | 0.15766248791730356  |
+-------------+----------------------+


## Случайный лес

In [30]:
model_evaluation('Random forest',X_Test=X, y_pred=RFRegr_y_pred, y_true=y)

+-------------------------------------+
|            Random forest            |
+-------------+-----------------------+
|   Field 1   |        Field 2        |
+-------------+-----------------------+
|      R2     |   0.9665229961684865  |
| Adjusted R2 |   0.9664429842090422  |
|     MSE     | 0.0010212898416656602 |
|     RMSE    |  0.03195762572009473  |
+-------------+-----------------------+


## XGBoost

In [31]:
model_evaluation('XGBoost',X_Test=X, y_pred=GBRegr_y_pred, y_true=y)

+-------------------------------------+
|               XGBoost               |
+-------------+-----------------------+
|   Field 1   |        Field 2        |
+-------------+-----------------------+
|      R2     |   0.9830170253144221  |
| Adjusted R2 |   0.9829764350307568  |
|     MSE     | 0.0005181031019065818 |
|     RMSE    |  0.02276187825963802  |
+-------------+-----------------------+


Сохранение моделей

In [29]:
import pickle

# Save Logistic Regression model
with open('logistic_regression_model.pkl', 'wb') as f:
    pickle.dump(LReg, f)

# Save Random Forest model
with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(RFRegr, f)

# Save XGBoost model
with open('xgboost_model.pkl', 'wb') as f:
    pickle.dump(GBRegr, f)

Основные выводы:

Линейная регрессия:

1. Модель объясняет лишь небольшую долю вариации целевой переменной , о чём свидетельствует значение коэффициента детерминации R² = 0.185 . Это указывает на то, что модель имеет ограниченную способность воспроизводить изменения в данных.
2. Скорректированный R² (0.183) также находится на низком уровне**, что подтверждает слабое качество модели даже с учетом количества используемых признаков.
3. Ошибка предсказания (MSE = 0.0249, RMSE = 0.1577) является относительно высокой, особенно если шкала целевой переменной невелика. Это говорит о том, что в среднем прогнозы модели отличаются от реальных значений на ~0.16 условных единиц.
4. Модель демонстрирует слабую предсказательную способность , поэтому использование данной модели для точного прогнозирования нецелесообразно без дополнительной доработки.


Случайный лес:

1. Модель демонстрирует высокое качество предсказания , о чём свидетельствует значение коэффициента детерминации R² = 0.967 . Это означает, что модель объясняет около 96.7% дисперсии целевой переменной , что говорит о её высокой объясняющей способности.
2. Скорректированный R² (0.966) также находится на высоком уровне**, что подтверждает стабильность модели и отсутствие значительного переобучения даже с учётом количества используемых признаков.
3. Ошибка предсказания невелика : MSE = 0.00102 и RMSE = 0.032 , что указывает на то, что в среднем прогнозы модели очень близки к реальным значениям. Это особенно важно, если шкала целевой переменной относительно мала.
4. Модель показывает отличную предсказательную способность и может быть рекомендована к использованию в практических задачах без существенной доработки.


XGBoost:

1. Модель демонстрирует очень высокое качество предсказания , что подтверждается значением коэффициента детерминации R² = 0.983 . Это означает, что модель объясняет около 98.3% дисперсии целевой переменной , что говорит о её исключительной способности воспроизводить изменения в данных.
2. Скорректированный R² (0.983) также находится на высоком уровне**, что указывает на стабильность модели и отсутствие существенного переобучения даже с учётом количества признаков.
3. Ошибка предсказания крайне мала : MSE = 0.00052 и RMSE = 0.023 , что свидетельствует о том, что прогнозы модели очень близки к реальным значениям . Такие показатели особенно важны в задачах, где требуется высокая точность предсказаний.
4. Модель XGBoost показывает отличную предсказательную способность и может быть рекомендована к использованию в производственной или аналитической практике без существенных доработок.
